# PySpark 2: reading, SQL and cleaning

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/spark/pyspark-02-reading-sql-and-cleaning.ipynb)

Most pipelines start by reading files and making them trustworthy. This module reads data with explicit schemas, queries it through Spark SQL and temp views, and cleans columns with built-in functions.

**You will learn to:**

- Read CSV and JSON with explicit schemas instead of inference
- Register temp views and mix DataFrame code with SQL
- Standardize strings, timestamps and types with column functions
- Wrap reusable cleaning logic in plain Python functions

**Before you start:** PySpark 1.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own. The setup cell installs PySpark when it is missing and starts a local session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

In [ ]:
import json
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import broadcast

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- nothing has executed yet")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()


orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)
read_schema = next(line for line in plan.splitlines() if line.startswith("ReadSchema"))
assert "GreaterThan(amount,0.00)" in plan  # the filter reached the Parquet reader
assert "status" not in read_schema  # unused columns are never read

## How do you read data with explicit schemas in PySpark?

Explicit schemas make pipelines safer and faster: inference can change with a small sample,
a malformed record, or a new partition. The cell below lands a small JSON file - one row
delivers `amount` as `"N/A"` - and reads it with a declared schema and a corrupt-record column.

In [ ]:
import json

raw_dir = os.path.join(WORK_DIR, "raw", "orders")
os.makedirs(raw_dir, exist_ok=True)
raw_rows = [
    {"order_id": "o1", "customer_id": "c1", "order_ts": "2026-10-04T09:15:00", "status": "paid", "amount": 120.00},
    {"order_id": "o2", "customer_id": "c2", "order_ts": "2026-10-04T10:05:00", "status": "shipped", "amount": 84.50},
    {"order_id": "o3", "customer_id": "c2", "order_ts": "2026-10-04T11:40:00", "status": "paid", "amount": "N/A"},
]
with open(os.path.join(raw_dir, "part-0.json"), "w") as handle:
    handle.write("\n".join(json.dumps(row) for row in raw_rows))

order_schema = T.StructType([
    T.StructField("order_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("amount", T.DecimalType(12, 2), True),
    T.StructField("_corrupt_record", T.StringType(), True),
])

orders_raw = (
    spark.read
    .schema(order_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(raw_dir)
    .cache()  # Spark refuses queries that touch only the corrupt-record column of an uncached read
)

good_orders = orders_raw.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
bad_orders = orders_raw.filter(F.col("_corrupt_record").isNotNull())
print("good:", good_orders.count(), "corrupt:", bad_orders.count())
bad_orders.select("_corrupt_record").show(truncate=False)
assert good_orders.count() == 2 and bad_orders.count() == 1

For CSV, be explicit about headers, delimiters, quotes, and escapes:

In [ ]:
csv_dir = os.path.join(WORK_DIR, "raw", "customers")
os.makedirs(csv_dir, exist_ok=True)
with open(os.path.join(csv_dir, "part-0.csv"), "w") as handle:
    handle.write(
        "customer_id,email,created_at\n"
        'c1,"ada@example.com",2026-01-03 09:00:00\n'
        'c2,"grace, hopper@example.org",2026-01-04 10:30:00\n'
    )

customers_csv = (
    spark.read
    .schema("customer_id string, email string, created_at timestamp")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(csv_dir)
)
customers_csv.show(truncate=False)
assert customers_csv.where("customer_id = 'c2'").first()["email"] == "grace, hopper@example.org"

How it works under the hood. Reading JSON without a schema launches an extra job that samples files to infer types, so the data is touched twice. An explicit `StructType` skips that pass and lets the reader decode straight into the declared types. `mode` controls corrupt records: `PERMISSIVE` keeps the raw text in a corrupt-record column, `DROPMALFORMED` drops rows silently, and `FAILFAST` raises.

Worked example. Schema inference on a 2 TB JSON dataset costs one extra scan before the real job runs. If the last of three files delivers `amount` as `"N/A"`, inference over a sample can settle on `string`, while the explicit `decimal(12,2)` schema keeps the contract and turns the bad row into a countable null or corrupt record.

Common mistake: a mismatched type under `PERMISSIVE` produces nulls rather than an error, so always count the corrupt-record column (or nulls) after a load.

## How do you use Spark SQL and temp views from PySpark?

PySpark and Spark SQL work together. DataFrames can be registered as temporary views, and SQL results come back as DataFrames.

In [ ]:
orders.createOrReplaceTempView("orders")

daily_sql = spark.sql("""
select
  to_date(order_ts) as order_date,
  count(*) as order_count,
  sum(amount) as revenue
from orders
where status in ('paid', 'shipped')
group by to_date(order_ts)
""")

# the SQL and the DataFrame versions are the same plan, so they return the same rows
sql_rows = sorted((r["order_date"], r["revenue"]) for r in daily_sql.collect())
df_rows = sorted((r["order_date"], r["revenue"]) for r in daily_revenue.collect())
assert sql_rows == df_rows
daily_sql.orderBy("order_date").show()

SQL can create tables too. On Unity Catalog you would write
`create or replace table main.gold.daily_revenue as ...`; the local session catalog stores a
Parquet table:

In [ ]:
spark.sql("drop table if exists gold_daily_revenue")
spark.sql("""
create table gold_daily_revenue using parquet as
select to_date(order_ts) as order_date, sum(amount) as revenue
from orders
where status in ('paid', 'shipped')
group by to_date(order_ts)
""")
assert spark.table("gold_daily_revenue").count() == daily_sql.count()
print(spark.catalog.listTables())

Interview point: PySpark DataFrame code and Spark SQL usually compile into the same execution engine. Choose the style that is clearer for the team and workload.

How it works under the hood. `createOrReplaceTempView` registers a name in the session catalog that points at the DataFrame's logical plan. `spark.sql(...)` parses SQL into the same kind of plan tree, so both paths meet before Catalyst optimization; a DataFrame and its SQL twin usually produce identical physical plans. A global temp view lives under `global_temp` and is visible across sessions in the same application.

Worked example. Register `orders` and run the DataFrame aggregate and the SQL aggregate side by side; both `explain()` outputs show the same `HashAggregate + Exchange` shape because they compile to the same plan.

Common mistake: temp views are session-scoped and disappear with the session, so a job that re-registers a stale plan under the same name can silently query old logic. Drop the view or use a fresh name when the underlying frame changes.

## How do you clean and standardize data in PySpark?

Use built-in column functions instead of Python row loops. Built-ins run inside Spark's optimized execution engine.

In [ ]:
raw_customers = spark.createDataFrame(
    [
        ("  Ada@Example.COM ", " us", "2026-01-03 09:00:00", "+1 (312) 555-0100"),
        ("grace@example.org", "gb ", "2026-01-04 10:30:00", "020 7946 0018"),
        ("not-an-email", "us", "2026-01-05 11:00:00", None),
    ],
    "email string, country_code string, created_at string, phone string",
)

clean_customers = (
    raw_customers
    .select(
        F.trim(F.lower("email")).alias("email"),
        F.upper(F.trim("country_code")).alias("country_code"),
        F.to_timestamp("created_at").alias("created_at"),
        F.regexp_replace("phone", r"[^0-9]", "").alias("phone_digits"),
    )
    .withColumn("email_domain", F.element_at(F.split("email", "@"), -1))
    .where(F.col("email").contains("@"))
)
clean_customers.show(truncate=False)

rows = {r["email"]: r for r in clean_customers.collect()}
assert set(rows) == {"ada@example.com", "grace@example.org"}
assert rows["ada@example.com"]["phone_digits"] == "13125550100"
assert rows["grace@example.org"]["country_code"] == "GB"

For reusable business logic, wrap DataFrame transformations in plain Python functions:

In [ ]:
def standardize_orders(df):
    return (
        df
        .withColumn("status", F.lower(F.trim("status")))
        .withColumn("order_date", F.to_date("order_ts"))
        .where(F.col("order_id").isNotNull())
    )


standardized = standardize_orders(orders)
assert standardized.count() == orders.count() and "order_date" in standardized.columns

How it works under the hood. Built-in string and timestamp functions are Catalyst expressions evaluated on Tungsten rows inside the JVM; `regexp_replace` becomes a Java regex call per row with no Python involved. Wrapping transformations in plain Python functions is fine because those functions run once on the driver to build the plan, not once per row.

Worked example:

```text
"  Ada@Example.COM "  -> lower(trim(...))        -> "ada@example.com"
"+1 (312) 555-0100"   -> regexp_replace([^0-9])   -> "13125550100"
```

Common mistake: reaching for `F.udf` to trim, lowercase or split. The built-in version is shorter, faster, and keeps whole-stage code generation intact.

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- Explicit schemas make reads faster and stop silent type changes.
- A cast that quietly returns null is a data-quality bug. Check nulls after every cast.
- Temp views and DataFrames compile to the same plan, so use whichever reads better.

**Next:** [PySpark 3: joins and aggregations](pyspark-03-joins-and-aggregations.ipynb)